# Week 15 V4: Manual Evaluation of Retrieval Impact

## Approach
1. Generate answers WITH and WITHOUT retrieval
2. Save raw answers in a clean format
3. Have Claude manually evaluate correctness against actual source code

No automated verification - human/LLM judgment for accuracy.

In [1]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 39.1 MB/s eta 0:00:00


In [2]:
import os
import json
import torch
import torch.nn.functional as F
from typing import List, Dict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [3]:
# Clone httpx
!git clone --depth 1 https://github.com/encode/httpx.git /content/httpx 2>/dev/null || true

def load_source_files(repo_dir: str) -> Dict[str, str]:
    files = {}
    for root, _, filenames in os.walk(os.path.join(repo_dir, 'httpx')):
        for fname in filenames:
            if fname.endswith('.py'):
                fpath = os.path.join(root, fname)
                rel_path = os.path.relpath(fpath, repo_dir)
                with open(fpath, 'r', encoding='utf-8') as f:
                    files[rel_path] = f.read()
    return files

SOURCE_FILES = load_source_files('/content/httpx')
print(f"Loaded {len(SOURCE_FILES)} source files")

Loaded 23 source files


In [4]:
# Load model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Models loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded!


## Tasks - Clear Questions with Known Answers

Each task includes the relevant file so we can verify against actual source.

In [5]:
TASKS = [
    # Timeout class
    {
        'id': 1,
        'question': 'What are ALL the parameters of httpx.Timeout.__init__()? List each parameter name.',
        'relevant_file': 'httpx/_config.py',
    },
    {
        'id': 2,
        'question': 'What is the default timeout value in httpx (the DEFAULT_TIMEOUT_CONFIG)? Give the exact number in seconds.',
        'relevant_file': 'httpx/_config.py',
    },

    # Limits class
    {
        'id': 3,
        'question': 'What are the parameters of httpx.Limits.__init__()? List each parameter name.',
        'relevant_file': 'httpx/_config.py',
    },
    {
        'id': 4,
        'question': 'What is the default value for max_connections in httpx.Limits?',
        'relevant_file': 'httpx/_config.py',
    },

    # Client parameters
    {
        'id': 5,
        'question': 'List the first 5 parameters of httpx.Client.__init__() in order (after self).',
        'relevant_file': 'httpx/_client.py',
    },

    # Exceptions
    {
        'id': 6,
        'question': 'What is the parent class of ConnectError in httpx._exceptions?',
        'relevant_file': 'httpx/_exceptions.py',
    },
    {
        'id': 7,
        'question': 'List ALL exception classes defined in httpx._exceptions module that inherit from HTTPStatusError or its subclasses.',
        'relevant_file': 'httpx/_exceptions.py',
    },
    {
        'id': 8,
        'question': 'What exception does httpx raise when the URL scheme is not http or https?',
        'relevant_file': 'httpx/_exceptions.py',
    },

    # Auth classes
    {
        'id': 9,
        'question': 'List ALL authentication classes defined in httpx._auth module.',
        'relevant_file': 'httpx/_auth.py',
    },

    # Decoders
    {
        'id': 10,
        'question': 'List ALL decoder classes in httpx._decoders module.',
        'relevant_file': 'httpx/_decoders.py',
    },

    # Response class
    {
        'id': 11,
        'question': 'What are 5 @property methods on httpx.Response class?',
        'relevant_file': 'httpx/_models.py',
    },

    # Specific implementation
    {
        'id': 12,
        'question': 'What is the value of DEFAULT_MAX_REDIRECTS in httpx?',
        'relevant_file': 'httpx/_config.py',
    },

    # Transport classes
    {
        'id': 13,
        'question': 'What transport classes are defined in httpx._transports.base module?',
        'relevant_file': 'httpx/_transports/base.py',
    },

    # URL parsing
    {
        'id': 14,
        'question': 'What class in httpx._urls represents a URL? What are its main attributes?',
        'relevant_file': 'httpx/_urls.py',
    },

    # Status codes
    {
        'id': 15,
        'question': 'How does httpx represent HTTP status codes? What class or structure is used?',
        'relevant_file': 'httpx/_status_codes.py',
    },
]

print(f"Created {len(TASKS)} tasks")

Created 15 tasks


## Build Retrieval Index

In [6]:
def create_chunks(source_files: Dict[str, str], chunk_size: int = 80) -> List[Dict]:
    chunks = []
    for filepath, content in source_files.items():
        lines = content.split('\n')
        for i in range(0, len(lines), chunk_size // 2):
            chunk_lines = lines[i:i + chunk_size]
            if len(chunk_lines) < 10:
                continue
            chunks.append({
                'filepath': filepath,
                'start_line': i,
                'content': '\n'.join(chunk_lines),
            })
    return chunks

CHUNKS = create_chunks(SOURCE_FILES)
print(f"Created {len(CHUNKS)} chunks")

chunk_texts = [f"{c['filepath']}:\n{c['content']}" for c in CHUNKS]
CHUNK_EMBEDDINGS = embedder.encode(chunk_texts, show_progress_bar=True, convert_to_tensor=True)
print(f"Embeddings: {CHUNK_EMBEDDINGS.shape}")

Created 224 chunks


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings: torch.Size([224, 384])


In [7]:
def retrieve(query: str, top_k: int = 3, target_file: str = None) -> str:
    """Retrieve relevant source code chunks."""
    query_emb = embedder.encode(query, convert_to_tensor=True)
    sims = F.cosine_similarity(query_emb.unsqueeze(0), CHUNK_EMBEDDINGS)

    if target_file:
        for i, chunk in enumerate(CHUNKS):
            if target_file in chunk['filepath']:
                sims[i] += 0.3

    top_idx = torch.topk(sims, k=min(top_k, len(CHUNKS))).indices

    parts = []
    for idx in top_idx:
        chunk = CHUNKS[idx.item()]
        parts.append(f"# From {chunk['filepath']} (line {chunk['start_line']}):\n{chunk['content']}")

    return "\n\n".join(parts)

In [8]:
def generate_answer(prompt: str, context: str = None, max_tokens: int = 400) -> str:
    """Generate answer with optional context."""
    if context:
        full_prompt = f"""[INST] Use this source code to answer the question:

{context}

Question: {prompt}

Answer based ONLY on the source code above. Be specific and precise. [/INST]"""
    else:
        full_prompt = f"""[INST] Answer this question about the httpx Python library:

{prompt}

Be specific and precise. [/INST]"""

    input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

## Run Experiment - Collect Raw Answers

In [9]:
%%time

results = []

for task in TASKS:
    print(f"\n{'='*60}")
    print(f"[Task {task['id']}] {task['question'][:60]}...")

    # Without retrieval
    answer_none = generate_answer(task['question'])

    # With retrieval
    context = retrieve(task['question'], top_k=3, target_file=task.get('relevant_file'))
    answer_ret = generate_answer(task['question'], context=context)

    results.append({
        'id': task['id'],
        'question': task['question'],
        'relevant_file': task['relevant_file'],
        'answer_without_retrieval': answer_none,
        'answer_with_retrieval': answer_ret,
        'retrieved_context': context[:1000] + '...' if len(context) > 1000 else context,
    })

    print(f"\nWithout retrieval:")
    print(answer_none[:300])
    print(f"\nWith retrieval:")
    print(answer_ret[:300])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



[Task 1] What are ALL the parameters of httpx.Timeout.__init__()? Lis...

Without retrieval:
The `httpx.Timeout` class has the following parameters in its `__init__` method:

1. `connect`: The maximum amount of time to wait when establishing a connection.
2. `read`: The maximum amount of time to wait for a server to send data.
3. `write`: The maximum amount of time to wait to send data to t

With retrieval:
Based on the source code, the parameters of httpx.Timeout.__init__() are:

* timeout: TimeoutTypes | UnsetType
* connect: None | float | UnsetType
* read: None | float | UnsetType
* write: None | float | UnsetType
* pool: None | float | UnsetType

[Task 2] What is the default timeout value in httpx (the DEFAULT_TIME...

Without retrieval:
The default timeout value in httpx is 30 seconds. This is the value of the `DEFAULT_TIMEOUT_CONFIG` constant, which is used as the default value for the `timeout` parameter in many of the methods in the `httpx` library.

Here is the relevant code 

## Save Results for Manual Evaluation

In [10]:
# Save results
output = {
    'experiment': 'Week15_V4_Manual_Evaluation',
    'model': MODEL_NAME,
    'num_tasks': len(TASKS),
    'results': results,
}

with open('week15_v4_raw_answers.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Saved to week15_v4_raw_answers.json")
print("\nDownload this file and share with Claude for manual evaluation.")

Saved to week15_v4_raw_answers.json

Download this file and share with Claude for manual evaluation.


## Display All Answers for Review

In [11]:
# Print all answers in a format easy to review
print("="*80)
print("ALL ANSWERS FOR MANUAL EVALUATION")
print("="*80)

for r in results:
    print(f"\n{'='*80}")
    print(f"TASK {r['id']}: {r['question']}")
    print(f"Relevant file: {r['relevant_file']}")
    print("-"*40)
    print("WITHOUT RETRIEVAL:")
    print(r['answer_without_retrieval'])
    print("-"*40)
    print("WITH RETRIEVAL:")
    print(r['answer_with_retrieval'])
    print()

ALL ANSWERS FOR MANUAL EVALUATION

TASK 1: What are ALL the parameters of httpx.Timeout.__init__()? List each parameter name.
Relevant file: httpx/_config.py
----------------------------------------
WITHOUT RETRIEVAL:
The `httpx.Timeout` class has the following parameters in its `__init__` method:

1. `connect`: The maximum amount of time to wait when establishing a connection.
2. `read`: The maximum amount of time to wait for a server to send data.
3. `write`: The maximum amount of time to wait to send data to the server.
4. `pool`: The maximum amount of time to wait for a connection to be released back to the pool.
5. `total`: The maximum amount of time to wait for the entire request to complete.

These parameters can be used to set different timeouts for different aspects of the request, such as the time to establish a connection, the time to read data from the server, the time to send data to the server, and the time to release a connection back to the pool.
-----------------------

## Extract Actual Source Code for Verification

Print the relevant source code sections so we can verify answers.

In [12]:
# Print relevant source sections
relevant_files = set(t['relevant_file'] for t in TASKS)

print("RELEVANT SOURCE CODE SECTIONS")
print("="*80)

for filepath in relevant_files:
    if filepath in SOURCE_FILES:
        content = SOURCE_FILES[filepath]
        print(f"\n{'='*80}")
        print(f"FILE: {filepath}")
        print(f"{'='*80}")
        # Print first 200 lines or whole file if shorter
        lines = content.split('\n')[:200]
        for i, line in enumerate(lines, 1):
            print(f"{i:4d} | {line}")
        if len(content.split('\n')) > 200:
            print(f"\n... (truncated, {len(content.split(chr(10)))} total lines)")

RELEVANT SOURCE CODE SECTIONS

FILE: httpx/_config.py
   1 | from __future__ import annotations
   2 | 
   3 | import os
   4 | import typing
   5 | 
   6 | from ._models import Headers
   7 | from ._types import CertTypes, HeaderTypes, TimeoutTypes
   8 | from ._urls import URL
   9 | 
  10 | if typing.TYPE_CHECKING:
  11 |     import ssl  # pragma: no cover
  12 | 
  13 | __all__ = ["Limits", "Proxy", "Timeout", "create_ssl_context"]
  14 | 
  15 | 
  16 | class UnsetType:
  17 |     pass  # pragma: no cover
  18 | 
  19 | 
  20 | UNSET = UnsetType()
  21 | 
  22 | 
  23 | def create_ssl_context(
  24 |     verify: ssl.SSLContext | str | bool = True,
  25 |     cert: CertTypes | None = None,
  26 |     trust_env: bool = True,
  27 | ) -> ssl.SSLContext:
  28 |     import ssl
  29 |     import warnings
  30 | 
  31 |     import certifi
  32 | 
  33 |     if verify is True:
  34 |         if trust_env and os.environ.get("SSL_CERT_FILE"):  # pragma: nocover
  35 |             ctx = ssl.